# Step 2 - Labelled training set and final model

Builds the labelled set MMS1 + THA + THB + C1 and trains the final model: an ensemble of HistGradientBoosting classifiers (max_iter=404, max_leaf_nodes=342), one per feature set given by `choose_best_features()` (feature sets with precision > 0.83 and recall > 0.821 in `all_precisions.pkl` / `all_recalls.pkl`); the prediction is the mean probability.

Inputs: `labelled_MMS1_THA_THB_2025-06-10.pkl` (MMS1, THA, THB labelled points), `C1_interesting_for_BL_extended.pkl` and the C1 label catalogues of 2025-08-04.

Outputs: `{OUT}/labelled_MMS1_THA_THB_C1.pkl` and `{OUT}/model_MMS_THEMIS_Cluster.pkl`.

**Note**: the model used in step 3 is the frozen file `/home/ghisalberti/GradientBoosting/diagnostics/model_MMS_THEMIS_Cluster.pkl`. Retraining does not give exactly the same model (HistGradientBoosting uses a random validation split for early stopping), and the exact training file of 2025-09-15 is lost: the file `labelled_MMS1_THA_THB_C1.pkl` was rewritten on 2026-04-13 with only 66432 of the 82515 C1 rows, and the code that wrote it is lost. Here the dated file of cell 18 is used as the training set.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # makes the blxline package importable
from blxline.paths import OUT, MODEL_LINES  # all outputs of this repo are written under OUT (never over the original files)

In [ ]:
# [from GradientBoosting/GradientBoostingBL_Cluster.ipynb, cell 0]
from blxline.catalogues import catalogue_to_edges_df
import pandas as pd
from blxline.model import compute_scores, recall_precision_curve, choose_best_features
import numpy as np
import warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier as GBC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from datetime import timedelta, datetime
import random as rd
from sklearn.metrics import auc
import seaborn as sns
from spok.models import planetary
from spok.plot import planet_env
from blxline.model import get_all_feature_combinaisons_multiple_choices, find_best_threshold, get_features_from_choice, get_all_feature_choices, get_name_choice, temporal_split, compute_scores, add_scores, x_y_set
from sklearn.ensemble import HistGradientBoostingClassifier as Hgbc
from spok.coordinates.coordinates import cartesian_to_spherical
from blxline.catalogues import catalogue_to_edges_df
import warnings
from spok.models.planetary import mp_shue1998_tangents, mp_shue1998_normal
from blxline.model import temporal_split, x_y_set, add_scores, compute_scores


def get_features_from_choice(choice, mandatory, features_yes_or_no, features_multiple_choice):
    features = list(np.array(mandatory).copy())
    for i, c in enumerate(choice):
        if i < len(features_yes_or_no):
            if c == '1':
                features += features_yes_or_no[i]
        else:
            c = int(c)
            features += features_multiple_choice[i - len(features_yes_or_no)][c]
    return features


## Labelled set

In [ ]:
# [from GradientBoosting/GradientBoostingBL_Cluster.ipynb, cell 2]
#labelled = catalogue_to_edges_df('/home/ghisalberti/catalogues/labelled_BL_C1.json', np.timedelta64(5,'s'))
#labels = catalogue_to_edges_df('/home/ghisalberti/catalogues/label_BL_C1.json', np.timedelta64(5,'s'))
#labelled = catalogue_to_edges_df('/home/ghisalberti/catalogues/labelled_BL_C1_2025-07-29.json', np.timedelta64(5,'s'))
#labels = catalogue_to_edges_df('/home/ghisalberti/catalogues/label_BL_C1_2025-07-29.json', np.timedelta64(5,'s'))

#date = '2025-07-31'
date = '2025-08-04'
labelled = catalogue_to_edges_df(f'/home/ghisalberti/catalogues/labelled_BL_C1_{date}.json', np.timedelta64(5,'s'))
labels = catalogue_to_edges_df(f'/home/ghisalberti/catalogues/label_BL_C1_{date}.json', np.timedelta64(5,'s'))
labels2 = catalogue_to_edges_df(f'/home/ghisalberti/catalogues/label_BL_C1_2_{date}.json', np.timedelta64(5,'s'))
labels = pd.concat([labels,labels2])

data = pd.read_pickle('/DATA/ghisalberti/Datasets/Cluster/C1/C1_interesting_for_BL_extended.pkl')
#pred = pd.read_pickle('/home/ghisalberti/GradientBoosting/all_pred/C1_pred_common_model_2025-02-25.pkl')
pred = pd.read_pickle('/home/ghisalberti/GradientBoosting/all_pred/C1_pred_common_model_2025-05-06.pkl')

data['labelled_data'] = 0
for i in range(len(labelled)):
    data.loc[labelled.iloc[i].begin:labelled.iloc[i].end,'labelled_data'] = 1

data['label_BL'] = 0
for i in range(len(labels)):
    data.loc[labels.iloc[i].begin:labels.iloc[i].end,'label_BL'] = 1
data['label_BL'] = data['label_BL'].values*data['labelled_data'].values

indices = data.index.values[data.index.isin(pred.index)]
data['pred_BL'] = np.nan
data.loc[indices, 'pred_BL'] = pred.loc[indices].values

In [ ]:
# [from GradientBoosting/GradientBoostingBL_Cluster.ipynb, cell 11]
labelled_mms_th = pd.read_pickle('/home/ghisalberti/make_datasets/labelled_MMS1_THA_THB_2025-06-10.pkl')
#labelled_mms_th = pd.read_pickle('/home/ghisalberti/make_datasets/labelled_MMS1_THA_THB.pkl')
len(labelled_mms_th[labelled_mms_th.sat.values != 'MMS1']), labelled_mms_th[labelled_mms_th.sat.values != 'MMS1'].label_BL.values.sum()

In [ ]:
# [from GradientBoosting/GradientBoostingBL_Cluster.ipynb, cell 15]
data = data[data.labelled_data.values==1]
data['sat'] = 'C1'
labelled = pd.concat([labelled_mms_th, data]).sort_index()
labelled

In [ ]:
# [RECONSTRUCTED: lost step. Bn_MP, Btan1_MP, Btan2_MP (mandatory features) on all labelled points, with the code of TrainingCommonModelChatGPT cell 10 (lines inside the docstring). Checked: same values as labelled_MMS1_THA_THB_C1_2025-08-05.pkl to 1e-14]
from spok.models.planetary import mp_shue1998_tangents, mp_shue1998_normal
subdata = labelled

# Compute MP frame features
x_normal, y_normal, z_normal = mp_shue1998_normal(subdata.theta.values, subdata.phi.values)
[x_tan1, y_tan1, z_tan1], [x_tan2, y_tan2, z_tan2] = mp_shue1998_tangents(subdata.theta.values, subdata.phi.values)

subdata['Bn_MP'] = x_normal * subdata.Bx.values + y_normal * subdata.By.values + z_normal * subdata.Bz.values
subdata['Btan1_MP'] = x_tan1 * subdata.Bx.values + y_tan1 * subdata.By.values + z_tan1 * subdata.Bz.values
subdata['Btan2_MP'] = x_tan2 * subdata.Bx.values + y_tan2 * subdata.By.values + z_tan2 * subdata.Bz.values
labelled = subdata

In [ ]:
# [from GradientBoosting/GradientBoostingBL_Cluster.ipynb, cell 18]
labelled.to_pickle(f'{OUT}/labelled_MMS1_THA_THB_C1.pkl')

## Final model

In [ ]:
# [from GradientBoosting/GradientBoostingBL_Cluster.ipynb, cell 131]
# pred massive on MMS for model with also Cluster data

warnings.filterwarnings("ignore")
kwargs = {'model_verbose': False, 'verbose': True, 'model': 'HGBC', 'freq_split': timedelta(days=3)}

subdata = pd.read_pickle(f'{OUT}/labelled_MMS1_THA_THB_C1.pkl')
all_features = choose_best_features()


gbcs, scalers = [], []
for columns in all_features:
    scaler = StandardScaler()
    scaler.fit(subdata[columns].values)
    scalers += [scaler]
    
    xtrain, ytrain = x_y_set(subdata, scaler, columns)

    gb = Hgbc(verbose=False, max_iter=404, max_leaf_nodes=342)
    gb.fit(xtrain, ytrain)
    gbcs += [gb]
model_results = {'all_features':all_features, 'models':gbcs, 'scalers':scalers}  
pd.to_pickle(model_results, f'{OUT}/model_MMS_THEMIS_Cluster.pkl')


from blxline.model import pred_ensemble
for sat in ['MMS1']:
    interesting = pd.read_pickle(f'/DATA/ghisalberti/Datasets/MMS/{sat}/{sat}_interesting_for_BL.pkl')
    pred = pred_ensemble(interesting, model_results)
    interesting['pred']=pred
    pred = pd.DataFrame(pred, index = interesting.index.values, columns=['pred_BL'])
    pred.to_pickle(f'{OUT}/all_pred/pred_{sat}_all_sat_model.pkl')